# Compact residual attention for v008

Definitions only. Execute the unchanged `research/sequence_components_v005.ipynb` code cells first, then these cells in the same namespace. Defining this notebook reads no assignment data and launches no training. The original scalar head, preprocessing, masked L1, optimizer, split, schedule, and main training engine remain caller owned.

The single architectural change is a residual attention branch over the final recurrent states: LayerNorm, a joint Q/K/V projection to **64 channels**, **four heads of width 16**, full-breath scaled dot-product attention, and a 64-to-encoder-width output projection. No feed-forward sublayer, causal mask, positional feature, new target feature, or attention dropout is added. The encoder already has ordered bidirectional context. All 80 known input positions may consult one another because complete offline input breaths are available. Attention is computed separately inside each breath with explicit `torch.bmm` and softmax; no scaled-dot-product-attention/FlashAttention API is used. The implementation uses PyTorch 2.2-compatible operations and never weakens deterministic settings.

The approved compact package remains BiLSTM128x2 with recurrent dropout .15 and the original 256-to-32-to-1 scalar head. This branch adds **66,496 parameters** to that package. `V8AttentionPressureRNN(spec, n_features)` also accepts other scalar v005 encoder sizes for synthetic engine checks; caller orchestration must enforce the approved real-data recipe. Existing R/C embedding/encoder options keep their v005 meaning, and `rc_heads=True` is rejected.

The constructor initializes all original modules before adding the branch, preserving every common tensor under the same starting RNG state. The attention output projection starts at zero, making the initial prediction exactly the original scalar prediction for identical encoder dropout draws. All original head dimensions and state-dict names remain intact. `load_baseline_state_dict(state)` explicitly transfers an independently retained original initialization with `strict=False` and rejects any missing common key, unexpected key, or shape mismatch; only attention keys may be missing. The constructor uses that same guarded transfer. Output-projection parameters learn on the first update; Q/K/V and LayerNorm obtain nonzero gradients after the projection opens. This is a deliberate stable initialization, not a frozen branch. No extra stochastic operation is introduced, although a different architecture need not follow the same long training trajectory.

Public APIs are `v8_attention_model(spec, n_features)`, `V8AttentionPressureRNN`, `v8_attention_selfcheck(device='cpu')`, and `v8_attention_source_guard(repository_root='.', expected_notebook_sha256=None)`. The factory accepts exactly the v005 model-spec schema and returns scalar standardized predictions `[breaths,80]` from transformed known inputs `[breaths,80,F+1]`; it accepts no labels. Use a pluggable engine factory rather than overwriting global v005 definitions. Source guarding checks the unchanged v005 notebook hash plus this notebook's implementation-cell hash, metadata/name/version, and optionally the caller's externally frozen complete-notebook SHA256. Call the guard explicitly before a production run; no files are read during definitions.

The optional selfcheck uses only generated tensors and restores caller RNG/backend settings. It verifies the approved package's shapes/counts, unchanged common initialization/head/dropout, exact baseline initialization, branch gradient opening, an independent manual-attention oracle, deterministic repeatability, breath isolation, save/load continuation, input-only forward signature, and rejected bad inputs/state transfer. CPU checks do not establish CUDA determinism, actual 256-breath throughput, or the 11 GB memory envelope; run the checks, exact engine-resume proof, and paired synthetic profiling on the actual course GPU before real-data fitting.


In [ ]:
# Required unchanged engine and frozen component identity. Definitions only.
_V8_ATTENTION_REQUIRED = ('V5PressureRNN', 'canonical_model_spec', 'seed_sequence',
    '_sequence_device', '_rng_state', '_restore_rng', 'torch', 'nn', 'np',
    'copy', 'hashlib', 'json', 'Path', 'V5_STEPS')
_missing_v8_attention = [name for name in _V8_ATTENTION_REQUIRED if name not in globals()]
if _missing_v8_attention:
    raise RuntimeError('Execute unchanged v005 engine definitions first: ' +
        ', '.join(_missing_v8_attention))
V8_ATTENTION_VERSION = 'v008-residual-attention-qkv64x4-v1'
V8_ATTENTION_NOTEBOOK = 'research/attention_components_v008.ipynb'
V8_ATTENTION_BASE_NOTEBOOK = 'research/sequence_components_v005.ipynb'
V8_ATTENTION_BASE_SHA256 = 'd20e92eec5889b094a87486f876c3e1a8ae1d7eb2aa096fa5dd530b64c914b4e'
V8_ATTENTION_HEADS = 4
V8_ATTENTION_WIDTH = 64
V8_ATTENTION_CODE_SHA256 = '144f87a2eb6cb31e946d1d09472982580f3f146f05719791c86653bbf9dfff41'


def v8_attention_source_guard(repository_root='.', expected_notebook_sha256=None):
    """Explicit local source verification; never called while defining cells.

    The implementation hash covers the model and synthetic-check code cells in
    order. A caller-frozen full-file hash additionally locks bootstrap and prose.
    This checks local source bytes; the caller must execute those same bytes.
    """
    repository_root = Path(repository_root)
    path = repository_root / V8_ATTENTION_NOTEBOOK
    source = path.read_bytes()
    notebook = json.loads(source)
    identity = notebook.get('metadata', {}).get('attention_component', {})
    if identity != dict(version=V8_ATTENTION_VERSION, notebook=V8_ATTENTION_NOTEBOOK,
                        implementation_sha256=V8_ATTENTION_CODE_SHA256):
        raise ValueError('v008 attention notebook identity/hash metadata mismatch')
    implementation_cells = [cell for cell in notebook['cells']
        if cell.get('metadata', {}).get('v8_attention_implementation')]
    if len(implementation_cells) != 2:
        raise ValueError('v008 attention implementation-cell layout changed')
    payload = '\n\n'.join(''.join(cell['source']) for cell in implementation_cells)
    implementation_sha256 = hashlib.sha256(payload.encode()).hexdigest()
    if implementation_sha256 != V8_ATTENTION_CODE_SHA256:
        raise ValueError('v008 attention implementation SHA256 mismatch')
    base_path = repository_root / V8_ATTENTION_BASE_NOTEBOOK
    base_sha256 = hashlib.sha256(base_path.read_bytes()).hexdigest()
    if base_sha256 != V8_ATTENTION_BASE_SHA256:
        raise ValueError('v008 attention requires the unchanged v005 engine source')
    notebook_sha256 = hashlib.sha256(source).hexdigest()
    if expected_notebook_sha256 is not None and notebook_sha256 != expected_notebook_sha256:
        raise ValueError('v008 attention full-notebook SHA256 mismatch')
    return dict(version=V8_ATTENTION_VERSION, notebook=V8_ATTENTION_NOTEBOOK,
        notebook_sha256=notebook_sha256, implementation_sha256=implementation_sha256,
        base_notebook=V8_ATTENTION_BASE_NOTEBOOK, base_sha256=base_sha256,
        external_notebook_hash_checked=expected_notebook_sha256 is not None)


In [ ]:
# Deterministic manual attention; no labels enter this model or its factory.
class _V8ResidualAttention(nn.Module):
    def __init__(self, encoder_width):
        super().__init__()
        self.encoder_width = int(encoder_width)
        self.num_heads = V8_ATTENTION_HEADS
        self.head_width = V8_ATTENTION_WIDTH // self.num_heads
        self.norm = nn.LayerNorm(self.encoder_width)
        self.qkv = nn.Linear(self.encoder_width, 3 * V8_ATTENTION_WIDTH)
        self.output = nn.Linear(V8_ATTENTION_WIDTH, self.encoder_width)
        # Preserve exact original predictions. The output learns on update one;
        # Q/K/V and LayerNorm start learning after output weights become nonzero.
        nn.init.zeros_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, sequence):
        batch, steps, _ = sequence.shape
        packed = self.qkv(self.norm(sequence)).reshape(
            batch, steps, 3, self.num_heads, self.head_width)
        q, k, v = packed.permute(2, 0, 3, 1, 4).unbind(dim=0)
        q = q.contiguous().reshape(batch * self.num_heads, steps, self.head_width)
        k = k.contiguous().reshape(batch * self.num_heads, steps, self.head_width)
        v = v.contiguous().reshape(batch * self.num_heads, steps, self.head_width)
        logits = torch.bmm(q, k.transpose(1, 2)) * (self.head_width ** -.5)
        probabilities = torch.softmax(logits, dim=-1)
        context = torch.bmm(probabilities, v)
        context = context.reshape(batch, self.num_heads, steps, self.head_width)
        context = context.transpose(1, 2).contiguous().reshape(batch, steps, V8_ATTENTION_WIDTH)
        return self.output(context)


class V8AttentionPressureRNN(V5PressureRNN):
    """Original v005 scalar model plus one small complete-breath attention residual.

    All encoder/head parameters retain their original state-dict names and sizes.
    Input contains only transformed known features and the internal joint R/C code.
    No hidden state, memory, pressure labels, or other breaths enter the branch.
    """
    def __init__(self, model_spec, n_features):
        spec = canonical_model_spec(model_spec)
        if spec['rc_heads']:
            raise ValueError('v008 attention requires the original scalar head: rc_heads=False')
        # Initialize baseline FIRST so an identical starting seed preserves every
        # baseline tensor regardless of the added branch's random initialization.
        super().__init__(spec, n_features)
        baseline_state = self.state_dict()
        encoder_width = self.head[0].in_features
        self.attention = _V8ResidualAttention(encoder_width)
        self.load_baseline_state_dict(baseline_state)

    def load_baseline_state_dict(self, state_dict):
        """Guarded common initialization transfer; only new branch keys may be absent.

        This is an initialization helper, not an attention-checkpoint loader.
        Use the ordinary strict load_state_dict for trained attention checkpoints.
        """
        expected = self.state_dict()
        common_keys = {name for name in expected if not name.startswith('attention.')}
        supplied_keys = set(state_dict)
        if supplied_keys != common_keys:
            missing = sorted(common_keys - supplied_keys)
            unexpected = sorted(supplied_keys - common_keys)
            raise ValueError(f'Baseline initialization keys changed: missing={missing}, '
                             f'unexpected={unexpected}')
        for name in common_keys:
            if not torch.is_tensor(state_dict[name]) or state_dict[name].shape != expected[name].shape:
                raise ValueError(f'Baseline initialization tensor shape changed: {name}')
        incompatible = self.load_state_dict(state_dict, strict=False)
        added_keys = sorted(name for name in expected if name.startswith('attention.'))
        if sorted(incompatible.missing_keys) != added_keys or incompatible.unexpected_keys:
            raise RuntimeError('Only new attention parameters may be absent from baseline state')
        return dict(common_tensor_count=len(common_keys), added_state_keys=added_keys)

    def encode(self, x):
        # Matches the original V5PressureRNN.forward encoder, including dropout.
        if x.ndim != 3 or x.shape[1] != V5_STEPS or x.shape[-1] != self.n_features + 1:
            raise ValueError('Expected complete known input breaths [B,80,F+1] with internal R/C code')
        joint = x[..., -1].long()
        sequence = x[..., :self.n_features]
        if self.joint_embedding is not None:
            sequence = torch.cat((sequence, self.joint_embedding(joint)), dim=-1)
        if not self.explicit_stack:
            sequence, _ = self.rnn(sequence)
        else:
            outputs = []
            for i, layer in enumerate(self.recurrent_layers):
                previous = sequence
                sequence, _ = layer(previous)
                if self.spec['residual']:
                    sequence = sequence + self.skip_projections[i](previous)
                outputs.append(sequence)
                if i + 1 < len(self.recurrent_layers):
                    sequence = self.between_layers(sequence)
            if self.spec['output_concat']:
                sequence = torch.cat(outputs, dim=-1)
        return sequence

    def forward(self, x):
        encoded = self.encode(x)
        return self.head(encoded + self.attention(encoded)).squeeze(-1)


def v8_attention_model(model_spec, n_features):
    """Pluggable two-argument scalar-model factory; no engine function is replaced."""
    return V8AttentionPressureRNN(model_spec, n_features)


In [ ]:
# Explicit synthetic proof only; defining this function starts no execution.
def v8_attention_selfcheck(device='cpu'):
    """Check actual compact architecture using generated known inputs only.

    Restores RNG, thread count and deterministic/TF32 settings even on failure.
    This is a small numerical proof, not the batch256 profiling or engine-resume
    gate, both of which caller orchestration must run on the actual GPU.
    """
    import inspect
    import platform

    target_device = _sequence_device(device)
    generator = torch.Generator().manual_seed(8008)
    rng_before = _rng_state(generator)
    backend_before = dict(threads=torch.get_num_threads(),
        deterministic=torch.are_deterministic_algorithms_enabled(),
        warn_only=torch.is_deterministic_algorithms_warn_only_enabled(),
        cudnn_benchmark=torch.backends.cudnn.benchmark,
        cudnn_deterministic=torch.backends.cudnn.deterministic,
        matmul_tf32=torch.backends.cuda.matmul.allow_tf32,
        cudnn_tf32=torch.backends.cudnn.allow_tf32)
    checks = {}

    def check(name, condition):
        if not bool(condition):
            raise AssertionError(name)
        checks[name] = True

    def rejects(name, action):
        try:
            action()
        except (ValueError, TypeError, RuntimeError):
            checks[name] = True
        else:
            raise AssertionError(name + ' did not reject')

    try:
        seed = 20261102
        spec = canonical_model_spec(dict(family='lstm', hidden=128, layers=2,
            dropout=.15, bidirectional=True, head_width=32))
        seed_sequence(seed)
        baseline = V5PressureRNN(spec, 47).to(target_device)
        seed_sequence(seed)
        model = v8_attention_model(spec, 47).to(target_device)
        baseline_state = baseline.state_dict()
        common_state = {name: value for name, value in model.state_dict().items()
                        if not name.startswith('attention.')}
        check('all_common_initial_tensors_exact', baseline_state.keys() == common_state.keys()
            and all(torch.equal(value, common_state[name]) for name, value in baseline_state.items()))
        transfer = model.load_baseline_state_dict(baseline_state)
        check('guarded_nonstrict_transfer_only_attention_missing',
            len(transfer['added_state_keys']) == 6 and transfer['common_tensor_count'] == len(baseline_state))
        check('original_scalar_head_dimensions_preserved',
            model.head[0].in_features == 256 and model.head[0].out_features == 32
            and model.head[-1].out_features == 1)
        check('original_recurrent_dropout_015_only', model.explicit_stack
            and model.between_layers.p == .15
            and [(name, module.p) for name, module in model.named_modules()
                 if isinstance(module, nn.Dropout)] == [('between_layers', .15)])
        baseline_parameters = sum(p.numel() for p in baseline.parameters())
        parameter_count = sum(p.numel() for p in model.parameters())
        added_parameters = parameter_count - baseline_parameters
        check('four_heads_fixed64_channels', model.attention.num_heads == 4
            and model.attention.head_width == 16 and model.attention.qkv.out_features == 192)
        check('compact_added_parameters_66496', added_parameters == 66496)
        check('zero_initial_output_projection',
            torch.count_nonzero(model.attention.output.weight) == 0
            and torch.count_nonzero(model.attention.output.bias) == 0)

        # Generated standardized input features plus internal R/C code only.
        x = torch.randn(3, V5_STEPS, 48, device=target_device)
        x[..., -1] = torch.tensor([0., 4., 8.], device=target_device)[:, None]
        model.eval(); baseline.eval()
        with torch.no_grad():
            prediction = model(x)
            check('finite_scalar_breath_output', prediction.shape == (3, 80)
                and torch.isfinite(prediction).all())
            check('exact_initial_baseline_prediction', torch.equal(prediction, baseline(x)))
            check('deterministic_eval_repeated_exact', torch.equal(prediction, model(x)))
        model.train(); baseline.train()
        dropout_state = torch.get_rng_state()
        cuda_dropout_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        with torch.no_grad():
            train_baseline = baseline(x)
            torch.set_rng_state(dropout_state)
            if cuda_dropout_state is not None:
                torch.cuda.set_rng_state_all(cuda_dropout_state)
            train_attention = model(x)
        check('same_dropout_draw_initial_predictions_exact', torch.equal(train_baseline, train_attention))

        # Two synthetic optimizer steps prove zero initialization does not freeze
        # attention: first the output opens, then Q/K/V and LayerNorm receive gradients.
        target = .2 * x[..., 2] + .1 * torch.sin(x[..., 3]) + .3
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=0.)
        first_projection_gradient = None
        for update in range(2):
            optimizer.zero_grad(set_to_none=True)
            loss = (model(x) - target).abs().mean()
            check(f'finite_loss_update_{update + 1}', torch.isfinite(loss))
            loss.backward()
            check(f'all_parameter_gradients_finite_update_{update + 1}', all(
                p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters()))
            if update == 0:
                first_projection_gradient = float(model.attention.output.weight.grad.abs().sum())
                check('output_projection_gradient_nonzero_first_update', first_projection_gradient > 0.)
                check('qkv_zero_gradient_only_at_initial_closed_projection',
                    torch.count_nonzero(model.attention.qkv.weight.grad) == 0)
            else:
                check('qkv_gradient_nonzero_second_update',
                    model.attention.qkv.weight.grad.abs().sum() > 0)
                check('layernorm_gradient_nonzero_second_update',
                    model.attention.norm.weight.grad.abs().sum() > 0)
                check('original_encoder_and_head_trainable',
                    model.recurrent_layers[0].weight_ih_l0.grad.abs().sum() > 0
                    and model.head[0].weight.grad.abs().sum() > 0)
            nn.utils.clip_grad_norm_(model.parameters(), 1.)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            encoded = model.encode(x)
            actual_branch = model.attention(encoded)
            normalized = model.attention.norm(encoded)
            packed = model.attention.qkv(normalized).reshape(3, 80, 3, 4, 16)
            q, k, v = packed.permute(2, 0, 3, 1, 4).unbind(dim=0)
            # Independent loop across breath/head, using ordinary matrix multiply.
            contexts = []
            for breath in range(3):
                heads = []
                for head in range(4):
                    scores = (q[breath, head] @ k[breath, head].transpose(0, 1)) / 4.
                    heads.append(torch.softmax(scores, dim=-1) @ v[breath, head])
                contexts.append(torch.cat(heads, dim=-1))
            expected_branch = model.attention.output(torch.stack(contexts))
            check('manual_bmm_matches_independent_attention_oracle',
                torch.allclose(actual_branch, expected_branch, rtol=1e-5, atol=1e-6))
            after = model(x)
            check('attention_branch_open_after_updates', actual_branch.abs().sum() > 0)
            check('deterministic_eval_after_updates_exact', torch.equal(after, model(x)))
            # Same batch size/order for exact comparisons, avoiding different
            # backend kernels for different batch sizes in an isolation check.
            altered_other_breath = x.clone()
            altered_other_breath[1, :, :47] += 2.
            isolated = model(altered_other_breath)
            check('other_breath_cannot_change_prediction', torch.equal(after[0], isolated[0])
                and torch.equal(after[2], isolated[2]))
            check('known_input_changes_own_prediction', not torch.equal(after[1], isolated[1]))
            model(altered_other_breath)
            check('no_persistent_breath_state', torch.equal(after, model(x)))
            clone = v8_attention_model(spec, 47).to(target_device)
            clone.load_state_dict(model.state_dict(), strict=True)
            clone.eval()
            check('strict_checkpoint_state_roundtrip_exact', torch.equal(after, clone(x)))

        # Only one input tensor is accepted. No label argument, label buffer or
        # pressure-dependent preprocessing is part of the prediction contract.
        check('forward_contract_known_inputs_only',
            tuple(inspect.signature(V8AttentionPressureRNN.forward).parameters) == ('self', 'x'))
        check('factory_contract_two_arguments',
            tuple(inspect.signature(v8_attention_model).parameters) == ('model_spec', 'n_features'))
        rejects('reject_pressure_label_argument', lambda: model(x, target))
        rejects('reject_incomplete_breath', lambda: model(x[:, :79]))
        rejects('reject_wrong_feature_count', lambda: model(x[..., :-1]))
        rejects('reject_categorical_rc_heads', lambda: v8_attention_model(dict(spec, rc_heads=True), 47))
        missing_state = dict(baseline_state)
        missing_state.pop(next(iter(missing_state)))
        rejects('reject_missing_common_initial_tensor', lambda: model.load_baseline_state_dict(missing_state))
        wrong_shape_state = dict(baseline_state)
        wrong_shape_state['head.0.weight'] = baseline_state['head.0.weight'][:1]
        rejects('reject_common_initial_tensor_shape_change',
            lambda: model.load_baseline_state_dict(wrong_shape_state))
        check('strict_determinism_stayed_enabled', torch.are_deterministic_algorithms_enabled()
            and not torch.is_deterministic_algorithms_warn_only_enabled())
        return dict(schema='v8_attention_selfcheck', version=V8_ATTENTION_VERSION,
            implementation_sha256=V8_ATTENTION_CODE_SHA256,
            data_scope='generated_synthetic_known_inputs_only; no assignment-data reads or fits',
            python_version=platform.python_version(), torch_version=str(torch.__version__),
            device=str(target_device), cuda_version=torch.version.cuda,
            cuda_name=torch.cuda.get_device_name(target_device) if target_device.type == 'cuda' else None,
            model_spec=spec, n_features=47, synthetic_shape=[3, 80, 48],
            baseline_parameters=int(baseline_parameters), parameter_count=int(parameter_count),
            added_parameters=int(added_parameters), attention_heads=4, attention_width=64,
            first_projection_gradient_l1=first_projection_gradient,
            checks=checks, check_count=len(checks), passed=all(checks.values()),
            gpu_profile_or_engine_resume_proof=False)
    finally:
        _restore_rng(rng_before, generator)
        torch.set_num_threads(backend_before['threads'])
        torch.use_deterministic_algorithms(backend_before['deterministic'],
            warn_only=backend_before['warn_only'])
        torch.backends.cudnn.benchmark = backend_before['cudnn_benchmark']
        torch.backends.cudnn.deterministic = backend_before['cudnn_deterministic']
        torch.backends.cuda.matmul.allow_tf32 = backend_before['matmul_tf32']
        torch.backends.cudnn.allow_tf32 = backend_before['cudnn_tf32']
